# Collections Analytics – Independent Business Analysis

## Objective

This analysis reconstructs collections and recovery performance from the available
operational data and independently evaluates the business claim:

> "Recovery has improved by 11% month-on-month."

The analysis focuses on data quality, payment validation, recovery performance,
portfolio risk, DPD distribution, and the reliability of the reported recovery trend.

## Business Questions

1. What actually happened to recovery performance?
2. Why did recovery performance change?
3. Is the reported 11% month-on-month improvement real?
4. What portfolio and delinquency characteristics affect recovery?
5. What are the key business implications and limitations?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}") 

In [5]:
import pyodbc
import pandas as pd

server = "TANUSH-LOQ"
database = "CollectionsAnalytics"

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

print("Database connection successful")

Database connection successful


In [4]:
%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [6]:
pd.read_sql("""
SELECT TOP 5 *
FROM clean.payments
""", conn)

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\900781605.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0002506,ACC0010762,BRW0004704,2026-08-07 18:21:57,TXN0000068724,"12,417.37",SUCCESS,CASH,VND0000005
1,PAYMENT0002508,ACC0028497,BRW0004291,2026-06-17 09:38:37,TXN0000015161,"91,888.56",SUCCESS,NETBANKING,VND0000003
2,PAYMENT0002509,ACC0001477,BRW0011973,2026-04-15 19:09:43,TXN0000027870,"46,954.08",SUCCESS,CASH,VND0000006
3,PAYMENT0002519,ACC0020115,BRW0005975,2026-04-10 19:31:04,TXN0000054550,"65,355.02",SUCCESS,NETBANKING,VND0000013
4,PAYMENT0002533,ACC0029099,BRW0004932,2026-06-25 06:24:38,TXN0000060793,"81,729.46",PENDING,UPI,VND0000010


In [7]:
# Load the analytical tables used in this analysis

clean_accounts = pd.read_sql("""
SELECT *
FROM clean.accounts
""", conn)

clean_payments = pd.read_sql("""
SELECT *
FROM clean.payments
""", conn)

gold_recovery = pd.read_sql("""
SELECT *
FROM gold.recovery_payments
""", conn)

clean_calls = pd.read_sql("""
SELECT *
FROM clean.calls
""", conn)

clean_whatsapp = pd.read_sql("""
SELECT *
FROM clean.whatsapp_events
""", conn)

print("Tables loaded successfully")
print("Clean Accounts:", clean_accounts.shape)
print("Clean Payments:", clean_payments.shape)
print("Gold Recovery:", gold_recovery.shape)
print("Clean Calls:", clean_calls.shape)
print("Clean WhatsApp:", clean_whatsapp.shape)

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\3204261743.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  clean_accounts = pd.read_sql("""
C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\3204261743.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  clean_payments = pd.read_sql("""
C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\3204261743.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  gold_recovery = pd.read_sql("""
C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\3204261743.py:18: UserWarning: pandas onl

Tables loaded successfully
Clean Accounts: (30000, 11)
Clean Payments: (25000, 9)
Gold Recovery: (17534, 8)
Clean Calls: (90079, 11)
Clean WhatsApp: (60000, 8)


In [8]:
# Basic structure and data-quality check

datasets = {
    "clean_accounts": clean_accounts,
    "clean_payments": clean_payments,
    "gold_recovery": gold_recovery,
    "clean_calls": clean_calls,
    "clean_whatsapp": clean_whatsapp
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing values:", df.isna().sum().sum())


clean_accounts
----------------------------------------
Rows: 30000
Columns: 11
Missing values: 455

clean_payments
----------------------------------------
Rows: 25000
Columns: 9
Missing values: 382

gold_recovery
----------------------------------------
Rows: 17534
Columns: 8
Missing values: 519

clean_calls
----------------------------------------
Rows: 90079
Columns: 11
Missing values: 1827

clean_whatsapp
----------------------------------------
Rows: 60000
Columns: 8
Missing values: 0


In [9]:
# Payment status distribution

payment_status = (
    clean_payments
    .groupby("payment_status")
    .agg(
        payment_records=("payment_id", "count"),
        total_amount=("amount", "sum")
    )
    .sort_values("total_amount", ascending=False)
)

payment_status

,payment_records,total_amount
payment_status,,
SUCCESS,17534,"1,315,583,965.60"
FAILED,3677,"278,418,566.11"
PENDING,2535,"190,223,176.36"
REVERSED,1254,"94,666,565.84"


### Payment Status Finding

The clean payment dataset contains 25,000 payment records.
SUCCESS represents the largest payment-status category, with 17534 records.
FAILED, PENDING and REVERSED transactions account for the remaining payment records.
This status-level analysis is important because recovery should be based on validated successful payment activity rather than treating all payment events as recovered amounts.

## 1. Payment Status Analysis

The cleaned payment dataset contains 25,000 payment records. The analysis separates transactions by payment status to distinguish successful payments from failed, pending and reversed transactions.

This is important because recovery performance should be evaluated using validated successful payments rather than treating every payment transaction as recovered cash.

In [10]:
# Calculate payment-status percentages

payment_status["record_pct"] = (
    payment_status["payment_records"]
    / payment_status["payment_records"].sum()
    * 100
).round(2)

payment_status

,payment_records,total_amount,record_pct
payment_status,,,
SUCCESS,17534,"1,315,583,965.60",70.14
FAILED,3677,"278,418,566.11",14.71
PENDING,2535,"190,223,176.36",10.14
REVERSED,1254,"94,666,565.84",5.02


### Finding

SUCCESS transactions account for 17,534 of the 25,000 cleaned payment records, representing approximately 70.14% of all payment records.
The remaining transactions consist of FAILED, PENDING and REVERSED statuses. Therefore, payment status must be considered when measuring actual recovery performance.

In [11]:
# Monthly validated recovery analysis

gold_recovery["event_at"] = pd.to_datetime(gold_recovery["event_at"])

monthly_recovery = (
    gold_recovery
    .groupby(gold_recovery["event_at"].dt.to_period("M"))
    .agg(
        successful_payment_count=("payment_id", "nunique"),
        validated_recovery=("amount", "sum")
    )
    .reset_index()
)

monthly_recovery["recovery_month"] = (
    monthly_recovery["event_at"].dt.to_timestamp()
)

monthly_recovery = monthly_recovery.drop(columns=["event_at"])

monthly_recovery["previous_month_recovery"] = (
    monthly_recovery["validated_recovery"].shift(1)
)

monthly_recovery["actual_mom_recovery_pct"] = (
    (
        monthly_recovery["validated_recovery"]
        - monthly_recovery["previous_month_recovery"]
    )
    / monthly_recovery["previous_month_recovery"]
    * 100
).round(2)

monthly_recovery

,successful_payment_count,validated_recovery,recovery_month,previous_month_recovery,actual_mom_recovery_pct
0,2464,"187,229,127.72",2026-01-01,NaN,NaN
1,2268,"170,142,453.88",2026-02-01,"187,229,127.72",-9.13
2,2524,"188,912,374.08",2026-03-01,"170,142,453.88",11.03
3,2406,"175,138,043.57",2026-04-01,"188,912,374.08",-7.29
4,2449,"184,250,278.79",2026-05-01,"175,138,043.57",5.20
5,2366,"175,559,727.09",2026-06-01,"184,250,278.79",-4.72
6,2441,"187,242,265.17",2026-07-01,"175,559,727.09",6.65
7,616,"47,109,695.30",2026-08-01,"187,242,265.17",-74.84


## 2. Recovery Performance by Risk Segment

This analysis compares outstanding loan exposure and recovered amounts across risk segments. It helps identify whether recovery performance differs between HIGH, MEDIUM, LOW and NPA segments.

In [12]:
# Recovery performance by risk segment

risk_recovery = pd.read_sql("""
SELECT
    a.risk_segment,
    COUNT(DISTINCT a.account_id) AS account_count,
    SUM(a.outstanding_amount) AS total_outstanding,
    COALESCE(SUM(r.recovered_amount), 0) AS total_recovered
FROM clean.accounts a
LEFT JOIN (
    SELECT
        account_id,
        SUM(amount) AS recovered_amount
    FROM gold.recovery_payments
    GROUP BY account_id
) r
    ON a.account_id = r.account_id
GROUP BY a.risk_segment
ORDER BY total_outstanding DESC
""", conn)

risk_recovery

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\1129546838.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  risk_recovery = pd.read_sql("""


,risk_segment,account_count,total_outstanding,total_recovered
0,HIGH,7552,"2,646,182,598.02","331,598,234.45"
1,LOW,7513,"2,633,311,102.30","330,857,833.45"
2,MEDIUM,7533,"2,628,179,462.74","330,510,084.16"
3,NPA,7402,"2,581,362,203.27","322,617,813.54"


In [13]:
# Calculate recovery percentage

risk_recovery["recovery_pct"] = (
    risk_recovery["total_recovered"]
    / risk_recovery["total_outstanding"]
    * 100
).round(2)

risk_recovery

,risk_segment,account_count,total_outstanding,total_recovered,recovery_pct
0,HIGH,7552,"2,646,182,598.02","331,598,234.45",12.53
1,LOW,7513,"2,633,311,102.30","330,857,833.45",12.56
2,MEDIUM,7533,"2,628,179,462.74","330,510,084.16",12.58
3,NPA,7402,"2,581,362,203.27","322,617,813.54",12.50


## 3. Recovery Performance by DPD Bucket

Days Past Due (DPD) is used to classify accounts according to delinquency severity. This analysis compares outstanding exposure and recovery across DPD buckets to identify where the largest collection opportunities exist.

DPD buckets are:
- CURRENT: 0 days
- 1-30: 1 to 30 days
- 31-60: 31 to 60 days
- 61-90: 61 to 90 days
- 90+: More than 90 days

In [14]:
# DPD bucket analysis

dpd_analysis = pd.read_sql("""
SELECT
    CASE
        WHEN dpd = 0 THEN 'CURRENT'
        WHEN dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN dpd BETWEEN 61 AND 90 THEN '61-90'
        WHEN dpd > 90 THEN '90+'
        ELSE 'UNKNOWN'
    END AS dpd_bucket,

    COUNT(*) AS account_count,

    SUM(outstanding_amount) AS total_outstanding

FROM clean.accounts

GROUP BY
    CASE
        WHEN dpd = 0 THEN 'CURRENT'
        WHEN dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN dpd BETWEEN 61 AND 90 THEN '61-90'
        WHEN dpd > 90 THEN '90+'
        ELSE 'UNKNOWN'
    END

ORDER BY total_outstanding DESC
""", conn)

dpd_analysis

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\1136579462.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dpd_analysis = pd.read_sql("""


,dpd_bucket,account_count,total_outstanding
0,1-30,10880,"3,834,100,526.98"
1,31-60,5514,"1,914,490,305.34"
2,90+,5453,"1,896,357,502.86"
3,61-90,5468,"1,895,868,400.31"
4,CURRENT,2685,"948,218,630.84"


In [15]:
# Calculate percentage of total outstanding exposure

dpd_analysis["outstanding_pct"] = (
    dpd_analysis["total_outstanding"]
    / dpd_analysis["total_outstanding"].sum()
    * 100
).round(2)

dpd_analysis

,dpd_bucket,account_count,total_outstanding,outstanding_pct
0,1-30,10880,"3,834,100,526.98",36.55
1,31-60,5514,"1,914,490,305.34",18.25
2,90+,5453,"1,896,357,502.86",18.08
3,61-90,5468,"1,895,868,400.31",18.07
4,CURRENT,2685,"948,218,630.84",9.04


In [16]:
# Recovery performance by DPD bucket

dpd_recovery = pd.read_sql("""
SELECT
    CASE
        WHEN a.dpd = 0 THEN 'CURRENT'
        WHEN a.dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN a.dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN a.dpd BETWEEN 61 AND 90 THEN '61-90'
        WHEN a.dpd > 90 THEN '90+'
        ELSE 'UNKNOWN'
    END AS dpd_bucket,

    COUNT(DISTINCT a.account_id) AS account_count,

    SUM(a.outstanding_amount) AS total_outstanding,

    COALESCE(SUM(r.recovered_amount), 0) AS total_recovered

FROM clean.accounts a

LEFT JOIN (
    SELECT
        account_id,
        SUM(amount) AS recovered_amount
    FROM gold.recovery_payments
    GROUP BY account_id
) r
    ON a.account_id = r.account_id

GROUP BY
    CASE
        WHEN a.dpd = 0 THEN 'CURRENT'
        WHEN a.dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN a.dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN a.dpd BETWEEN 61 AND 90 THEN '61-90'
        WHEN a.dpd > 90 THEN '90+'
        ELSE 'UNKNOWN'
    END

ORDER BY total_outstanding DESC
""", conn)

dpd_recovery

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\2527617051.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dpd_recovery = pd.read_sql("""


,dpd_bucket,account_count,total_outstanding,total_recovered
0,1-30,10880,"3,834,100,526.98","471,539,791.04"
1,31-60,5514,"1,914,490,305.34","252,777,082.91"
2,90+,5453,"1,896,357,502.86","235,069,503.75"
3,61-90,5468,"1,895,868,400.31","238,735,656.64"
4,CURRENT,2685,"948,218,630.84","117,461,931.26"


In [17]:
# Recovery rate by DPD bucket

dpd_recovery["recovery_pct"] = (
    dpd_recovery["total_recovered"]
    / dpd_recovery["total_outstanding"]
    * 100
).round(2)

dpd_recovery

,dpd_bucket,account_count,total_outstanding,total_recovered,recovery_pct
0,1-30,10880,"3,834,100,526.98","471,539,791.04",12.30
1,31-60,5514,"1,914,490,305.34","252,777,082.91",13.20
2,90+,5453,"1,896,357,502.86","235,069,503.75",12.40
3,61-90,5468,"1,895,868,400.31","238,735,656.64",12.59
4,CURRENT,2685,"948,218,630.84","117,461,931.26",12.39


### Finding

The 1–30 DPD bucket has the largest outstanding exposure at ₹3.83 billion, representing approximately 36.55% of total outstanding exposure. The 31–60 DPD bucket has the highest recovery rate at 13.20%, while the 1–30 DPD bucket has the lowest recovery rate at 12.30%.
Although the recovery rates are relatively close across DPD buckets, the large outstanding exposure in the 1–30 DPD bucket represents the largest collection opportunity by absolute exposure.

## 4. Risk Segment × DPD Analysis

This analysis combines risk segment and delinquency level to identify specific groups with high outstanding exposure. It helps prioritize collection efforts by showing where risk and delinquency overlap.

In [18]:
risk_dpd = pd.read_sql("""
SELECT
    risk_segment,

    CASE
        WHEN dpd = 0 THEN 'CURRENT'
        WHEN dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN dpd BETWEEN 61 AND 90 THEN '61-90'
        WHEN dpd > 90 THEN '90+'
        ELSE 'UNKNOWN'
    END AS dpd_bucket,

    COUNT(*) AS account_count,

    SUM(outstanding_amount) AS total_outstanding

FROM clean.accounts

GROUP BY
    risk_segment,
    CASE
        WHEN dpd = 0 THEN 'CURRENT'
        WHEN dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN dpd BETWEEN 61 AND 90 THEN '61-90'
        WHEN dpd > 90 THEN '90+'
        ELSE 'UNKNOWN'
    END

ORDER BY
    total_outstanding DESC
""", conn)

risk_dpd

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\2768585346.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  risk_dpd = pd.read_sql("""


,risk_segment,dpd_bucket,account_count,total_outstanding
0,MEDIUM,1-30,2819,"992,794,955.56"
1,HIGH,1-30,2754,"961,905,618.68"
2,NPA,1-30,2666,"945,771,216.26"
3,LOW,1-30,2641,"933,628,736.48"
4,LOW,31-60,1431,"498,772,265.06"
5,LOW,90+,1419,"495,880,492.30"
6,HIGH,90+,1391,"489,775,351.59"
7,MEDIUM,61-90,1363,"480,961,563.29"
8,LOW,61-90,1382,"479,165,713.22"
9,HIGH,61-90,1364,"477,910,200.90"


In [19]:
risk_dpd["exposure_pct"] = (
    risk_dpd["total_outstanding"]
    / risk_dpd["total_outstanding"].sum()
    * 100
).round(2)

risk_dpd

,risk_segment,dpd_bucket,account_count,total_outstanding,exposure_pct
0,MEDIUM,1-30,2819,"992,794,955.56",9.47
1,HIGH,1-30,2754,"961,905,618.68",9.17
2,NPA,1-30,2666,"945,771,216.26",9.02
3,LOW,1-30,2641,"933,628,736.48",8.90
4,LOW,31-60,1431,"498,772,265.06",4.76
5,LOW,90+,1419,"495,880,492.30",4.73
6,HIGH,90+,1391,"489,775,351.59",4.67
7,MEDIUM,61-90,1363,"480,961,563.29",4.59
8,LOW,61-90,1382,"479,165,713.22",4.57
9,HIGH,61-90,1364,"477,910,200.90",4.56


## 5. Key Business Findings

### Recovery Performance
- Total validated recovery is approximately ₹1.316 billion.
- There are 17,534 recovery transactions.
- 13,284 accounts and 7,987 borrowers were recovered.
- Overall recovery rate is approximately 12.54%.

### Monthly Recovery
- Recovery increased from February to March 2026 by approximately 11.03%.
- The largest month-on-month decline occurred in August 2026, with recovery decreasing by 74.84%.
- August should be interpreted carefully because it contains only partial-month data.

### Risk Segment
- Recovery rates across HIGH, LOW, MEDIUM and NPA segments are very similar, ranging approximately from 12.50% to 12.58%.
- Therefore, risk segment alone does not create a large difference in recovery performance.

### DPD Analysis
- The 1–30 DPD bucket has the largest outstanding exposure at approximately ₹3.834 billion.
- It represents approximately 36.55% of total outstanding exposure.
- The 31–60 DPD bucket has the highest recovery rate at approximately 13.20%.
- Recovery rates across DPD buckets remain relatively close, between approximately 12.30% and 13.20%.

### Risk × DPD
- MEDIUM-risk accounts in the 1–30 DPD bucket have the largest individual Risk × DPD exposure at approximately ₹992.79 million.
- The 1–30 DPD bucket is the largest exposure category across every risk segment.
- This indicates that early-stage delinquency is the largest collection opportunity by outstanding amount.

In [20]:
# Final portfolio-level validation

validation = pd.read_sql("""
SELECT
    COUNT(*) AS total_accounts,
    SUM(outstanding_amount) AS total_outstanding,
    SUM(principal_amount) AS total_principal,
    AVG(CAST(dpd AS FLOAT)) AS average_dpd
FROM clean.accounts
""", conn)

validation

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\2337130022.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  validation = pd.read_sql("""


,total_accounts,total_outstanding,total_principal,average_dpd
0,30000,"10,489,035,366.33","12,103,665,678.04",56.51


In [21]:
# Final recovery validation

recovery_validation = pd.read_sql("""
SELECT
    COUNT(*) AS recovery_transactions,
    COUNT(DISTINCT account_id) AS recovered_accounts,
    SUM(amount) AS total_recovery
FROM gold.recovery_payments
""", conn)

recovery_validation

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\3549782294.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  recovery_validation = pd.read_sql("""


,recovery_transactions,recovered_accounts,total_recovery
0,17534,13284,"1,315,583,965.60"


In [22]:
# Payment status validation

payment_validation = pd.read_sql("""
SELECT
    payment_status,
    COUNT(*) AS payment_records,
    SUM(amount) AS total_amount
FROM clean.payments
GROUP BY payment_status
ORDER BY total_amount DESC
""", conn)

payment_validation

C:\Users\Lnovo\AppData\Local\Temp\ipykernel_18812\2965929230.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payment_validation = pd.read_sql("""


,payment_status,payment_records,total_amount
0,SUCCESS,17534,"1,315,583,965.60"
1,FAILED,3677,"278,418,566.11"
2,PENDING,2535,"190,223,176.36"
3,REVERSED,1254,"94,666,565.84"
